# Approach 2 / NB1 — Data Prep (Whisper speech-to-phoneme)

**Approach 2 = the ANPLers method:** fine-tune a Whisper model (Tarteel's Quran
Whisper, or openai/whisper-*) to output **phoneme sequences** instead of text.
It keeps Whisper's seq2seq decoder; we simply train it on (audio -> phoneme string).
Because the IqraEval phonemes are plain space-separated symbols (e.g. `f ii h i nn`),
Whisper's existing BPE tokenizer can encode them directly — no new tokens needed.

This NB just (1) confirms the data and (2) optionally caches it to disk so NB2
never re-downloads (HF's CDN times out a lot). Run on **CPU**.

> If you already built the Approach-1 `iqra-cache` Kaggle Dataset, you can SKIP
> running this and just attach that cache in NB2 — it's the same data.

In [ ]:
!pip install -q "datasets==2.21.0" "transformers==4.44.2"

In [ ]:
import os, time
from datasets import load_dataset

TRAIN_REPO = "IqraEval/Iqra_train"
TTS_REPO   = "IqraEval/Iqra_TTS"
USE_TTS    = True
OUT_DIR    = "/kaggle/working/iqra_prep"
os.makedirs(OUT_DIR, exist_ok=True)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

# Optional HF token (Kaggle secret) for faster, less-timeout-prone downloads.
try:
    from kaggle_secrets import UserSecretsClient
    _t = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = _t
    print("HF_TOKEN loaded.")
except Exception as e:
    print("No HF_TOKEN (anonymous):", e)

In [ ]:
ds = load_dataset(TRAIN_REPO, split="train")
print(ds)
ex = ds[0]
for k in ds.column_names:
    if k != "audio":
        print(f"{k!r}: {ex[k]!r}")
print("audio sr:", ex["audio"]["sampling_rate"], "| n_samples:", len(ex["audio"]["array"]))
# We will train Whisper to produce the phoneme_ref string from the audio.

In [ ]:
# Cache to disk so NB2 needs no network (same idea as Approach 1).
SAVE_ARROW = True
if SAVE_ARROW:
    load_dataset(TRAIN_REPO).save_to_disk(os.path.join(OUT_DIR, "iqra_train_arrow"))
    print("saved train+dev ->", os.path.join(OUT_DIR, "iqra_train_arrow"))
    if USE_TTS:
        try:
            load_dataset(TTS_REPO).save_to_disk(os.path.join(OUT_DIR, "iqra_tts_arrow"))
            print("saved TTS ->", os.path.join(OUT_DIR, "iqra_tts_arrow"))
        except Exception as e:
            print("TTS save skipped:", e)
    print("DONE. Commit -> 'Create Dataset' -> attach to NB2 -> set LOCAL_*_DISK.")

### Output
- `iqra_prep/iqra_train_arrow/`, `iqra_prep/iqra_tts_arrow/` — cached datasets.
- Commit → Create Dataset → attach to NB2. (Or reuse the Approach-1 `iqra-cache`.)